In [ ]:
# core
import json
import urllib.request
import urllib.parse
import sqlite3
import pandas as pd
from pathlib import Path
import time
import sys
import os

# NLP 
import spacy
from transformers import pipeline

# topic modeling
from bertopic import BERTopic


: 

In [5]:
DB_PATH = Path("/Users/zhanghanshi/Documents/GitHub/dashboard_conflict_data/gnews_articles.db") # The path to your SQLite database
OUTPUT_CSV_FILE = "processed_articles_monthly.csv"

In [6]:
# Helping Functions

def load_data_from_db(db_path):
    """Loads all required columns from the SQLite database."""
    print(f"Loading data from database: {db_path}...")
    if not db_path.exists():
        print(f"Error: Database file not found at {db_path}")
        return pd.DataFrame()
    
    conn = sqlite3.connect(db_path)
    # We select all columns needed, excluding 'id' and 'url'
    query = """
    SELECT publishedAt, title, description, content, source_name
    FROM gnews_articles
    """
    try:
        df = pd.read_sql_query(query, conn)
        print(f"Successfully loaded {len(df)} articles.")
    except Exception as e:
        print(f"Database query failed: {e}")
        df = pd.DataFrame()
    finally:
        conn.close()
    
    return df

def preprocess_dates(df):
    """Converts publishedAt to YYYY-MM-DD format and renames the column."""
    # Convert to datetime objects first
    df['publishedAt'] = pd.to_datetime(df['publishedAt'], errors='coerce')
    # Extract only the date part and rename
    df['published_date'] = df['publishedAt'].dt.strftime('%Y-%m-%d')
    # Create Year-Month column for aggregation
    df['published_month'] = df['publishedAt'].dt.to_period('M').astype(str)
    
    df = df.drop(columns=['publishedAt'])
    return df

def run_translation_pipeline(df, pipeline_func, columns_to_translate):
    """Translates specified columns from German to English."""
    for col_de in columns_to_translate:
        col_en = f"{col_de}_en"
        print(f"Translating column '{col_de}' to '{col_en}'...")
        
        # Apply the translation function
        df[col_en] = df[col_de].fillna('').apply(
            lambda x: pipeline_func(x)[0]['translation_text'] if x.strip() else ''
        )
    return df

def analyze_emotion(text_en, pipeline_func):
    """Runs emotion analysis on English text using the bertweet model."""
    if not text_en or pipeline_func is None:
        return 'neutral', 0.0
    try:
        # bertweet model's labels are: 'anger', 'joy', 'optimism', 'sadness'
        result = pipeline_func(text_en[:512])[0]
        return result['label'], result['score']
    except Exception:
        return 'neutral', 0.0

def map_emotion_to_numeric(label, score):
    """Maps categorical emotion labels to a numeric score for visualization."""
    if label in ['joy', 'optimism']:
        return score
    if label in ['anger', 'sadness']:
        return -score
    return 0.0

# --- 2. Main Processing Block ---

def run_pipeline():
    # Load raw data
    df = load_data_from_db(DB_PATH)
    if df.empty:
        return
    
    # Preprocess dates
    df = preprocess_dates(df)
    
    # --- Load NLP Models ---
    print("\nLoading NLP models for translation and emotion analysis...")
    try:
        # Load Translation Pipeline (DE -> EN)
        translation_pipeline = pipeline("translation_de_to_en", 
                                        model="Helsinki-NLP/opus-mt-de-en")
        print("✅ Translation model loaded.")
        
        # Load Emotion Analysis Pipeline (EN)
        emotion_pipeline_en = pipeline("text-classification", 
                                       model="finiteautomata/bertweet-base-emotion-analysis")
        print("✅ Emotion model loaded.")
    except Exception as e:
        print(f"❌ Failed to load NLP models: {e}. Check network/dependencies.")
        return

    # --- Translation ---
    df = run_translation_pipeline(df, translation_pipeline, ['title', 'description', 'content'])

    # --- Create Analysis Text (Efficiency Choice) ---
    # We combine title and description for efficiency in NLP, instead of using long content field
    df['analysis_text_en'] = df['title_en'].fillna('') + " " + df['description_en'].fillna('')
    
    # --- Emotion Analysis ---
    print("\nRunning Emotion Analysis...")
    df[['sentiment_label', 'sentiment_score']] = df['analysis_text_en'].apply(
        lambda x: pd.Series(analyze_emotion(x, emotion_pipeline_en))
    )
    df['sentiment_numeric'] = df.apply(
        lambda row: map_emotion_to_numeric(row['sentiment_label'], row['sentiment_score']), axis=1
    )
    print("✅ Emotion Analysis complete.")
    
    # --- BERTopic Modeling (Monthly Clustering) ---
    # The monthly clustering requires significant memory and time.
    print("\nRunning BERTopic Modeling (Monthly Clustering)...")
    
    # Prepare Vectorizer (use English stopwords since we analyze EN text)
    # We use a default English list, as we are analyzing translated text
    vectorizer = CountVectorizer(stop_words='english', min_df=5) 
    
    # Prepare BERTopic
    topic_model = BERTopic(
        embedding_model="all-MiniLM-L6-v2", # Efficient English embedding model
        vectorizer_model=vectorizer,
        verbose=False
    )
    
    # Run BERTopic on the entire analysis text (for a general baseline)
    topics_general, _ = topic_model.fit_transform(df['analysis_text_en'].tolist())
    df['topic_id_general'] = topics_general
    
    # Get general topic info for the dashboard
    topic_info_general = topic_model.get_topic_info()
    topic_info_general.to_csv("topic_info_general.csv", index=False)
    
    print("✅ General Topic Modeling complete.")

    # --- Load (Save Final CSV) ---
    final_columns = [
        'published_date', 'published_month', 'source_name', 'topic_category',
        'title_en', 'description_en', 'content_en', 'analysis_text_en',
        'sentiment_label', 'sentiment_numeric', 'topic_id_general'
    ]
    df_final = df[[col for col in final_columns if col in df.columns]]
    df_final.to_csv(OUTPUT_CSV_FILE, index=False, encoding='utf-8-sig')
    
    print("\n--- Pipeline Success ---")
    print(f"Final data saved to: {OUTPUT_CSV_FILE}")

if __name__ == "__main__":
    run_pipeline()

Loading data from database: /Users/zhanghanshi/Documents/GitHub/dashboard_conflict_data/gnews_articles.db...
Database query failed: Execution failed on sql '
    SELECT publishedAt, title, description, content, source_name
    FROM gnews_articles
    ': no such table: gnews_articles
